# Database Validation
SQL queries against `war_games.db` to verify correctness.

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('war_games.db')
conn.execute('PRAGMA foreign_keys = ON')

def q(sql):
    return pd.read_sql_query(sql, conn)

## 1. Row counts

In [ ]:
for table in ['People', 'Teams', 'Salaries', 'Batting_with_WAR', 'Pitching_with_WAR']:
    count = q(f'SELECT COUNT(*) AS n FROM {table}').iloc[0, 0]
    print(f'{table}: {count:,} rows')

## 2. WAR coverage (should be 100%)

In [ ]:
q("""
SELECT
    'Batting' AS table_name,
    COUNT(*) AS total,
    SUM(CASE WHEN WAR IS NOT NULL THEN 1 ELSE 0 END) AS with_war,
    ROUND(100.0 * SUM(CASE WHEN WAR IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct
FROM Batting_with_WAR
UNION ALL
SELECT
    'Pitching',
    COUNT(*),
    SUM(CASE WHEN WAR IS NOT NULL THEN 1 ELSE 0 END),
    ROUND(100.0 * SUM(CASE WHEN WAR IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 1)
FROM Pitching_with_WAR
""")

## 3. Spot-check: Aaron Judge

In [ ]:
q("""
SELECT b.yearID, b.teamID, t.name AS team_name,
       b.G, b.AB, b.HR, b.RBI, b.WAR
FROM Batting_with_WAR b
JOIN Teams t ON b.teamID = t.teamID AND b.yearID = t.yearID
WHERE b.playerID = 'judgeaa01'
ORDER BY b.yearID
""")

## 4. Spot-check: Ohtani (batting + pitching)

In [ ]:
print('Batting:')
display(q("""
SELECT b.yearID, b.teamID, b.G, b.AB, b.HR, b.WAR
FROM Batting_with_WAR b
WHERE b.playerID = 'ohtansh01'
ORDER BY b.yearID
"""))

print('Pitching:')
display(q("""
SELECT p.yearID, p.teamID, p.G, p.W, p.L, p.ERA, p.WAR
FROM Pitching_with_WAR p
WHERE p.playerID = 'ohtansh01'
ORDER BY p.yearID
"""))

## 5. Multi-team player
Verify a traded player has separate rows per team with WAR.

In [ ]:
q("""
SELECT p.nameFirst || ' ' || p.nameLast AS name,
       b.yearID, b.stint, b.teamID, b.G, b.AB, b.HR, b.WAR
FROM Batting_with_WAR b
JOIN People p ON b.playerID = p.playerID
WHERE b.playerID IN (
    SELECT playerID FROM Batting_with_WAR
    WHERE yearID = 2024
    GROUP BY playerID, yearID
    HAVING COUNT(*) > 1
    LIMIT 1
)
AND b.yearID = 2024
ORDER BY b.stint
""")

## 6. Join integrity: Batting → People → Teams

In [ ]:
# Batting rows with no matching People record
orphan_people = q("""
SELECT COUNT(*) AS orphans FROM Batting_with_WAR b
LEFT JOIN People p ON b.playerID = p.playerID
WHERE p.playerID IS NULL
""")
print(f'Batting rows with no People match: {orphan_people.iloc[0,0]}')

# Batting rows with no matching Teams record
orphan_teams = q("""
SELECT COUNT(*) AS orphans FROM Batting_with_WAR b
LEFT JOIN Teams t ON b.teamID = t.teamID AND b.yearID = t.yearID
WHERE t.teamID IS NULL
""")
print(f'Batting rows with no Teams match: {orphan_teams.iloc[0,0]}')

## 7. Top 5 batting WAR per year

In [ ]:
q("""
SELECT yearID, name, teamID, HR, WAR FROM (
    SELECT b.yearID, p.nameFirst || ' ' || p.nameLast AS name,
           b.teamID, b.HR, b.WAR,
           RANK() OVER (PARTITION BY b.yearID ORDER BY b.WAR DESC) AS rnk
    FROM Batting_with_WAR b
    JOIN People p ON b.playerID = p.playerID
    WHERE b.yearID IN (2020, 2022, 2024)
)
WHERE rnk <= 5
ORDER BY yearID, WAR DESC
""")

## 8. Top 5 pitching WAR per year

In [ ]:
q("""
SELECT yearID, name, teamID, W, ERA, WAR FROM (
    SELECT p2.yearID, pe.nameFirst || ' ' || pe.nameLast AS name,
           p2.teamID, p2.W, p2.ERA, p2.WAR,
           RANK() OVER (PARTITION BY p2.yearID ORDER BY p2.WAR DESC) AS rnk
    FROM Pitching_with_WAR p2
    JOIN People pe ON p2.playerID = pe.playerID
    WHERE p2.yearID IN (2020, 2022, 2024)
)
WHERE rnk <= 5
ORDER BY yearID, WAR DESC
""")

## 9. Highest-paid vs highest WAR (2015)

In [ ]:
print('Top 10 paid batters (2015):')
display(q("""
SELECT p.nameFirst || ' ' || p.nameLast AS name,
       s.teamID, s.salary, b.WAR
FROM Salaries s
JOIN People p ON s.playerID = p.playerID
JOIN Batting_with_WAR b ON s.playerID = b.playerID
    AND s.yearID = b.yearID AND s.teamID = b.teamID
WHERE s.yearID = 2015
ORDER BY s.salary DESC
LIMIT 10
"""))

print('Top 10 WAR batters (2015):')
display(q("""
SELECT p.nameFirst || ' ' || p.nameLast AS name,
       b.teamID, b.WAR, s.salary
FROM Batting_with_WAR b
JOIN People p ON b.playerID = p.playerID
LEFT JOIN Salaries s ON b.playerID = s.playerID
    AND b.yearID = s.yearID AND b.teamID = s.teamID
WHERE b.yearID = 2015
ORDER BY b.WAR DESC
LIMIT 10
"""))